# BGL ablation campaign — Google Colab

Two flags in the launch cell:

- **`RUN_PREPARE`** — parse BGL, Azure-large enrich, build Family A graphs with `scripts/prepare_bgl_campaign.py`.
- **`RUN_TRAIN`** — `--mode train-only` on those graphs (Family A or B).

BGL graphs are **20 min / 10 min time windows** (`window_id`). Family A uses `configs/ablation_representation_bgl.yaml` (no HDFS `feature_contract_stabilized_v2` arm). Family B reuses `ablation_train.yaml` on `baseline_full`. Learning rate is 0.001 from `ablation_base.yaml`.

**Prepare on Drive / Colab secrets**

1. Upload `BGL_full.log` (~686 MB) to `MyDrive/hybrid-log-analyzer-artifacts/data/raw/bgl/BGL_full.log`.
2. Add notebook secrets (🔑): `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO`.
3. GPU runtime helps SBERT; parse/enrich are mostly CPU + API.

Colab uses `sentence-transformers` 5.x. **Do not mix** these graphs with a locally prepared BGL campaign (ST 2.2.2). HDFS prepare can still run locally in parallel.

**Families (train)**

- **A (representation)** — one rebuilt graph per arm (`tfidf_only`, `no_llm_enrichment`, …).
- **B (train)** — reuse `baseline_full` and sweep architecture/loss (`alpha_0`, `gine_mean_agg`, …).

See `notebooks/ABLATION.md`. Single-graph debug: `6_GAE_Training_BGL_Colab.ipynb`.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

In [ ]:
from pathlib import Path
import gzip
import json
import os
import shutil
import subprocess

REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer-research.git"
GIT_REF = "ablation-cursor"  # pin the campaign branch; do not pull mid-run
DATASET = "bgl"
CAMPAIGN_ID = "bgl_ablation_20260917"
RUN_PREPARE = True           # parse / enrich / build Family A graphs on this runtime
RUN_TRAIN = False            # GAE train-only after graphs exist
FAMILY = "B"                 # used when RUN_TRAIN: "A" or "B"
SMOKE = True                 # train only: 1 epoch / 5k graphs
REUSE_DRIVE_CACHE = True
AZURE_SECRET_KEYS = (
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO",
)

GRAPH_DATASET_RELATIVE_PATH = (
    f"campaigns/{CAMPAIGN_ID}/graphs/baseline_full/graph_dataset.pt.gz"
)
CAMPAIGN_RELATIVE_DIR = f"campaigns/{CAMPAIGN_ID}"
BGL_RAW_RELATIVE = "data/raw/bgl/BGL_full.log"


def find_local_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "modules" / "models" / "gae.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the hybrid-logs-analyzer-research checkout."
    )

REPO_ROOT = Path("/content/hybrid-logs-analyzer-research") if IN_COLAB else find_local_repository_root()
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")


def final_drive_sync() -> None:
    if not IN_COLAB:
        return
    for directory_name in ("artifacts", "models", "outputs", "runs", "campaigns"):
        source = WORKSPACE_ROOT / directory_name
        if source.exists():
            destination = DRIVE_ARTIFACT_ROOT / directory_name
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"], check=True)


def load_azure_credentials() -> None:
    missing = []
    userdata_get = None
    if IN_COLAB:
        from google.colab import userdata as colab_userdata

        userdata_get = colab_userdata.get
    for key in AZURE_SECRET_KEYS:
        if os.environ.get(key):
            continue
        value = None
        if userdata_get is not None:
            try:
                value = userdata_get(key)
            except Exception:
                value = None
        if value:
            os.environ[key] = str(value)
        else:
            missing.append(key)
    if missing:
        raise EnvironmentError(
            "Azure credentials missing: "
            + ", ".join(missing)
            + ". Add them as Colab secrets (🔑) with notebook access, or export them locally."
        )


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

    def stage_from_drive(relative_path: str) -> Path:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        destination = WORKSPACE_ROOT / relative_path
        if not source.exists():
            raise FileNotFoundError(f"Missing Drive artifact: {source}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source.is_dir():
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])
        else:
            shutil.copy2(source, destination)
        return destination

    def stage_tree_from_drive(relative_path: str) -> None:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        if source.exists():
            destination = WORKSPACE_ROOT / relative_path
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])

    def stage_bgl_raw_from_drive() -> Path:
        nested = DRIVE_ARTIFACT_ROOT / BGL_RAW_RELATIVE
        flat = DRIVE_ARTIFACT_ROOT / "data/raw/BGL_full.log"
        destination = WORKSPACE_ROOT / BGL_RAW_RELATIVE
        if nested.exists():
            return stage_from_drive(BGL_RAW_RELATIVE)
        if flat.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(flat, destination)
            return destination
        raise FileNotFoundError(
            "Upload BGL_full.log to Drive at "
            f"{nested} (preferred) or {flat}."
        )

    if REUSE_DRIVE_CACHE:
        stage_tree_from_drive(f"artifacts/cache/{DATASET}")
        stage_tree_from_drive("artifacts/runs")
        stage_tree_from_drive(f"outputs/{DATASET}")
        stage_tree_from_drive("campaigns")

    CAMPAIGN_DIR = None
    GRAPH_DATASET_PATH = None
    RAW_LOG_PATH = None
    if RUN_PREPARE:
        load_azure_credentials()
        RAW_LOG_PATH = stage_bgl_raw_from_drive()
        CAMPAIGN_DIR = WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR
    if RUN_TRAIN and not RUN_PREPARE:
        if FAMILY.upper() == "A":
            CAMPAIGN_DIR = stage_from_drive(CAMPAIGN_RELATIVE_DIR)
        else:
            campaign_source = DRIVE_ARTIFACT_ROOT / CAMPAIGN_RELATIVE_DIR / "manifest.json"
            if campaign_source.exists():
                CAMPAIGN_DIR = stage_from_drive(CAMPAIGN_RELATIVE_DIR)
            else:
                GRAPH_DATASET_PATH = stage_from_drive(GRAPH_DATASET_RELATIVE_PATH)
                if GRAPH_DATASET_PATH.suffix == ".gz":
                    unpacked = GRAPH_DATASET_PATH.with_suffix("")
                    print(f"Decompressing {GRAPH_DATASET_PATH.name}")
                    with gzip.open(GRAPH_DATASET_PATH, "rb") as src, open(unpacked, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
                    GRAPH_DATASET_PATH = unpacked
    if not RUN_PREPARE and not RUN_TRAIN:
        raise ValueError("Set RUN_PREPARE and/or RUN_TRAIN in the launch cell.")
else:
    CAMPAIGN_DIR = (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR) if (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR / "manifest.json").exists() else None
    GRAPH_DATASET_PATH = None
    RAW_LOG_PATH = WORKSPACE_ROOT / BGL_RAW_RELATIVE
    if RUN_PREPARE:
        load_azure_credentials()
        if not RAW_LOG_PATH.exists():
            raise FileNotFoundError(f"Missing BGL raw log: {RAW_LOG_PATH}")
        CAMPAIGN_DIR = WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR

print(f"Code checkout : {REPO_ROOT}")
print(f"Workspace     : {WORKSPACE_ROOT}")
print(f"Prepare       : {RUN_PREPARE}")
print(f"Train         : {RUN_TRAIN}")
print(f"Campaign dir  : {CAMPAIGN_DIR}")
print(f"Raw log       : {RAW_LOG_PATH}")
print(f"Family        : {FAMILY}")

In [ ]:
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT),
    ])
else:
    print("Local environment — use this repo venv / requirements.txt, not requirements-colab.txt.")

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

gae_module = REPO_ROOT / "src" / "modules" / "models" / "gae.py"
if not gae_module.exists():
    raise FileNotFoundError(
        f"Missing {gae_module}. This Colab clone of {GIT_REF!r} does not contain "
        "the GAE package. Commit and push src/modules/models/, then "
        "Runtime → Restart session and rerun from the clone cell."
    )

import torch
print(f"Working directory: {Path.cwd()}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## Prepare Family A graphs

Runs `scripts/prepare_bgl_campaign.py` into `/content/workspace/campaigns/<CAMPAIGN_ID>/` and checkpoints to Drive after each stage. Completed graph arms are skipped on resume. Set `RUN_PREPARE=False` to skip this cell.

This does **not** train. After it finishes, set `RUN_TRAIN=True` (keep `RUN_PREPARE=False` unless you want to rebuild) and run the train cell.

In [ ]:
if not RUN_PREPARE:
    print("Skipping prepare (RUN_PREPARE=False).")
else:
    prepare_command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "prepare_bgl_campaign.py"),
        "--campaign-id", CAMPAIGN_ID,
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--config", str(REPO_ROOT / "configs" / "ablation_base.yaml"),
        "--matrix", str(REPO_ROOT / "configs" / "ablation_representation_bgl.yaml"),
    ]
    if IN_COLAB:
        prepare_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])
    print(" ".join(str(part) for part in prepare_command))
    try:
        completed = subprocess.run(prepare_command, check=False)
    finally:
        final_drive_sync()
    if completed.returncode:
        raise RuntimeError(f"BGL prepare failed with exit code {completed.returncode}")
    CAMPAIGN_DIR = WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR
    GRAPH_DATASET_PATH = None
    print(f"Prepare finished. Campaign dir: {CAMPAIGN_DIR}")
    manifest_path = CAMPAIGN_DIR / "manifest.json"
    if manifest_path.exists():
        print(manifest_path.read_text()[:2000])

## Run train-only

Skipped when `RUN_TRAIN=False`. The CLI writes a uniform eval pack per arm under `outputs/bgl/<campaign_id>_<name>/` and a leaderboard under `outputs/bgl/campaigns/<campaign_id>/`. Completed arms are skipped on resume.

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "ablation_base.yaml"
if not RUN_TRAIN:
    print("Skipping train (RUN_TRAIN=False).")
else:
    FAMILY_KEY = FAMILY.upper()
    runner_command = [
        sys.executable,
        str(REPO_ROOT / "run_ablation.py"),
        "--mode", "train-only",
        "--config", str(BASE_CONFIG_PATH),
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--campaign-id", CAMPAIGN_ID,
        "--family", FAMILY_KEY,
        "--set", f"experiment.dataset={DATASET}",
    ]
    if SMOKE:
        runner_command.extend([
            "--set", "training.test_run=true",
            "--set", "training.epochs=1",
            "--set", "training.test_samples=5000",
        ])
    if FAMILY_KEY == "A":
        runner_command.extend([
            "--matrix", str(REPO_ROOT / "configs" / "ablation_representation_bgl.yaml"),
        ])
    if CAMPAIGN_DIR is not None and (Path(CAMPAIGN_DIR) / "manifest.json").exists():
        runner_command.extend(["--campaign-dir", str(CAMPAIGN_DIR)])
    elif FAMILY_KEY == "B":
        if GRAPH_DATASET_PATH is None:
            raise ValueError("Family B needs a campaign dir or GRAPH_DATASET_RELATIVE_PATH")
        runner_command.extend([
            "--matrix", str(REPO_ROOT / "configs" / "ablation_train.yaml"),
            "--graph-dataset", str(GRAPH_DATASET_PATH),
        ])
    else:
        raise ValueError("Family A requires campaigns/<id>/ on Drive (manifest.json + graphs/).")
    if IN_COLAB:
        runner_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])

    print(" ".join(str(part) for part in runner_command))
    try:
        completed = subprocess.run(runner_command, check=False)
    finally:
        final_drive_sync()

    if completed.returncode:
        raise RuntimeError(f"Campaign runner failed with exit code {completed.returncode}")
    print("Campaign train finished.")

## Leaderboard and comparison plots

Loaded from disk (not notebook RAM) so a later session can replay without retraining.

In [ ]:
import pandas as pd
from IPython.display import Image, display

report_root = DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT
campaign_report = report_root / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID
if not RUN_TRAIN and not (campaign_report / "leaderboard.csv").exists():
    print("No leaderboard yet. After prepare, set RUN_TRAIN=True and rerun train.")
else:
    if not (campaign_report / "leaderboard.json").exists():
        subprocess.check_call([
            sys.executable, str(REPO_ROOT / "run_ablation.py"),
            "--mode", "report",
            "--config", str(BASE_CONFIG_PATH),
            "--workspace-root", str(WORKSPACE_ROOT),
            "--code-root", str(REPO_ROOT),
            "--campaign-id", CAMPAIGN_ID,
            "--set", f"experiment.dataset={DATASET}",
        ])
        if IN_COLAB:
            final_drive_sync()
        campaign_report = WORKSPACE_ROOT / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID

    leaderboard = pd.read_csv(campaign_report / "leaderboard.csv")
    display(leaderboard)
    print((campaign_report / "README.md").read_text())
    for figure_name in ("ablation_comparison.png", "loss_curves.png", "component_comparison.png"):
        path = campaign_report / figure_name
        if path.exists():
            display(Image(filename=str(path)))

## Replay a previous campaign

Set `CAMPAIGN_ID` above and run only the leaderboard cell. Per-run packs live in `outputs/bgl/<campaign_id>_<arm>/figures/`.

In [ ]:
from IPython.display import Image, display

runs_root = (DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT) / "outputs" / DATASET
for run_dir in sorted(runs_root.glob(f"{CAMPAIGN_ID}_*")):
    if not (run_dir / "metrics.json").exists():
        continue
    metrics = json.loads((run_dir / "metrics.json").read_text())
    print(f"{run_dir.name}: F1={metrics.get('test_f1')} PR-AUC={metrics.get('test_pr_auc')} ROC-AUC={metrics.get('test_roc_auc')}")
    preview = run_dir / "figures" / "test_pr_roc.png"
    if preview.exists():
        display(Image(filename=str(preview)))